# Real Wikinews classification: sparse lexical features

This is a **notebook acceptance run, not a pytest test or a model benchmark**. We use the 15-article March 2025 Wikinews fixture already shipped with TeAL, including its locally assigned `topic_group` annotations. The research question is whether short news excerpts classified as *politics* can be distinguished from other subjects.

We build **count**, **TF-IDF**, and **TF-IDF + numerical metadata** representations, compare sklearn classifiers with TeAL's `ClassifierSearch`, refit the selected configuration on development labels, and inspect predictions against a protected audit set.

**Interpretation boundary:** The corpus is tiny, and lexical representations are fitted on the whole unlabeled corpus before splitting labels (a **transductive** setup). This demonstrates TeAL's finite-corpus workflow; the audit score is *not* a publishable generalization estimate. Never include `topic_group` or the target `is_politics` in predictor features.

Source: Wikinews article excerpts, CC BY 4.0; topic labels are TeAL fixture annotations, not Wikinews editorial labels. No network access is needed.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from text_analysis_lab import Project
from text_analysis_lab.analysis import ClassifierSearch
from text_analysis_lab.translators import CountVectorizer, TfidfTransformer, FeatureMatrix


def find_repo_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "tests/fixtures/wikinews_2025_small/articles.jsonl").exists():
            return root
    raise RuntimeError("Run this notebook inside the checked-out TeAL repository.")


ROOT = find_repo_root()
FIXTURE = ROOT / "tests/fixtures/wikinews_2025_small/articles.jsonl"
WORK = Path(tempfile.mkdtemp(prefix="teal-wikinews-lexical-"))
print("Source:", FIXTURE, "\nRun:", WORK)

In [ ]:
articles = pd.DataFrame([
    json.loads(line) for line in FIXTURE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]).sort_values("article_id").reset_index(drop=True)
assert len(articles) == 15
articles["is_politics"] = articles["topic_group"].eq("politics").astype(int)
articles["n_chars"] = articles["text"].str.len().astype(int)
articles["n_paragraphs"] = articles["text"].str.split("\n\n").str.len().astype(int)

display(articles[["article_id", "title", "topic_group", "is_politics"]])
display(articles["topic_group"].value_counts())
assert articles["is_politics"].sum() == 8
assert articles["is_politics"].nunique() == 2

## Register articles, then reserve audit labels

We split **TeAL keys**, not DataFrame row numbers. Only the development label artifact is supplied to search. We will not open the audit labels until the model and hyperparameters are finalized.

In [ ]:
csv_path = WORK / "wikinews.csv"
articles[["article_id", "text", "topic_group", "is_politics", "n_chars", "n_paragraphs"]].to_csv(csv_path, index=False)
project = Project.create(WORK / "project", name="wikinews_lexical_acceptance")
source = project.read_csv(
    csv_path,
    text_fields="text",
    metadata_fields=["article_id", "topic_group", "is_politics", "n_chars", "n_paragraphs"],
)
split = project.split(
    source,
    labels=("development", "audit"),
    proportions=(0.7, 0.3),
    random_state=2025,
    stratify="is_politics",
)
development, audit = split["development"], split["audit"]
print("Development:", development.n_rows, "Audit:", audit.n_rows)
assert development.n_rows + audit.n_rows == len(articles)
assert development.n_rows >= 8 and audit.n_rows >= 3

## Sparse representation families

The extra numerical variables, length and paragraph count, are computed from article text, not from labels. `FeatureMatrix` adds them lazily to the TF-IDF matrix rather than storing a second copy of the text features.

In [ ]:
counts = project.translate(
    CountVectorizer(text_field="text", stop_words="english",
                    ngram_range=(1, 2), max_features=300),
    source,
)["output"]
tfidf = project.translate(TfidfTransformer(norm="l2"), counts)["output"]
augmented = project.translate(
    FeatureMatrix(
        data_columns=True, metadata_columns=["n_chars", "n_paragraphs"],
        matrix_type="sparse",
    ), tfidf,
)["output"]
for label, representation in [("counts", counts), ("tfidf", tfidf), ("augmented", augmented)]:
    matrix = representation.get_matrix(positions=[0, 2])
    print(label, matrix.shape, "sparse?", sparse.issparse(matrix))
    assert sparse.issparse(matrix) and matrix.shape[0] == 2
assert augmented.get_matrix(data_columns=["n_chars"]).shape == (len(articles), 1)

## Model selection using development labels only

The same two CV folds are used for every representation–model candidate. The reported score reflects a very small sample and should be treated as an API/diagnostic exercise, not a substantive finding.

In [ ]:
search = ClassifierSearch(
    representations=[counts, tfidf, augmented],
    models={
        "logistic": {"C": [0.1, 1.0, 10.0]},
        "linear_svm": {"C": [0.5, 1.0]},
    },
    cv=2, scoring="f1", random_state=2025, batch_size=4,
)
plan = search.plan(development, target="is_politics")
print("Planned model fits:", int(plan["planned_fits"].sum()))
display(plan[["candidate", "representation_id", "model", "params", "planned_fits"]])

search.fit(development, target="is_politics")
results = search.results().sort_values(
    ["mean_score", "std_score"], ascending=[False, True]
).reset_index(drop=True)
display(results[["candidate", "representation_id", "model", "params",
                 "mean_score", "fold_scores", "fit_seconds", "status", "error"]])
assert results["status"].eq("ok").any()

## Fresh full-development refit; only now unlock audit labels

`finalize` fits a fresh estimator using **all development labels**. Predictions cover the original corpus, but scoring uses only the independent audit keys. We join by TeAL's primary key and inspect mistakes by title.

In [ ]:
winner = int(results.loc[results["status"] == "ok", "candidate"].iloc[0])
selected = search.finalize(candidate=winner)
assert selected.provenance["development_rows"] == development.n_rows
predicted_artifact = project.translate(
    selected.predictor, selected.source, batch_size=4,
)["output"]
print("Selected:", selected.provenance["model"], selected.provenance["params"])

# Protected audit annotations are accessed for the first time AFTER model selection.
gold = audit.query(
    key_columns=True, data_columns=False,
    metadata_columns=["is_politics", "topic_group", "article_id"],
    metadata_mode="full", form="table",
)
prediction_rows = predicted_artifact.query(
    key_columns=True, data_columns=["prediction"], form="table",
)
audit_results = gold.merge(
    prediction_rows,
    on=list(audit.primary_key),
    validate="one_to_one", how="left",
)
assert len(audit_results) == audit.n_rows
assert audit_results["prediction"].notna().all()
audit_results["correct"] = audit_results["is_politics"].astype(int) == audit_results["prediction"].astype(int)
display(audit_results[["article_id", "topic_group", "is_politics", "prediction", "correct"]])
print("Audit accuracy:", accuracy_score(audit_results["is_politics"], audit_results["prediction"]))
print("Audit F1:", f1_score(audit_results["is_politics"], audit_results["prediction"], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(
    audit_results["is_politics"], audit_results["prediction"], labels=[0, 1],
))


## Persistence and reproducibility

Reload the fitted TeAL operator from disk. Its training provenance contains development key identity and chosen hyperparameters, but does not prevent application to another compatible representation.

In [ ]:
prediction_id = predicted_artifact.artifact_id
source_id = selected.source.artifact_id
operation_id = predicted_artifact.descriptor["operation_id"]
operator_id = project.catalog.get_operation(operation_id)["operator_id"]
stored_predictions = prediction_rows["prediction"].to_numpy(copy=True)
project.close()

reopened = Project.open(WORK / "project")
frozen = reopened.get_operator(operator_id)
assert frozen.training_provenance == selected.provenance
replayed = reopened.translate(frozen, reopened.get_artifact(source_id))["output"]
after = replayed.query(
    key_columns=True, data_columns=["prediction"], form="table",
)
np.testing.assert_array_equal(after["prediction"].to_numpy(), stored_predictions)
print("Reloaded estimator matches saved predictions on all", len(after), "articles.")
reopened.close()

## Interpretation checklist

Inspect which rows were misclassified. Are those errors plausibly related to topic overlap, short excerpts or weak lexical signals? Did length features help, or did they merely dominate on this tiny corpus? Since unsupervised vocabulary/IDF were fitted before splitting, what would need to change for a truly prospective evaluation?

**Success means the notebook executes, the keys and frozen predictions agree, and the model-selection/audit boundary is explicit—not that the measured accuracy is high.**